In [3]:
%%bash
PREFIX=https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/05-monitoring
wget $PREFIX/rag_helper.py
wget $PREFIX/starter.py

--2026-07-20 18:03:59--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/05-monitoring/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1814 (1.8K) [text/plain]
Saving to: ‘rag_helper.py’

     0K .                                                     100% 3.47M=0s

2026-07-20 18:03:59 (3.47 MB/s) - ‘rag_helper.py’ saved [1814/1814]

--2026-07-20 18:03:59--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/05-monitoring/starter.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting respons

In [1]:
from starter import rag

query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

It keeps calling the model in a `while True` loop.

Each iteration:
- sends the full message history to the model,
- checks the response for any `function_call` items,
- runs those tools and appends the results to memory,
- sets a flag if any tool was called.

Then it stops only when there are **no function calls** in the response:

```python
if has_function_calls == False:
    break
```

So the model drives the process, and the code just repeats until the model returns a final message with no more tool calls.


In [14]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

Overriding of current TracerProvider is not allowed


In [2]:
from rag_helper import RAGBase

class RAGTraced(RAGBase):
    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search") as span:
            span.set_attribute("query", query)
            span.set_attribute("num_results", num_results)
            return super().search(query, num_results=num_results)

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            span.set_attribute("prompt_length", len(prompt))
            response = super().llm(prompt)

            usage = response.usage
            span.set_attribute("input_tokens", usage.input_tokens)
            span.set_attribute("output_tokens", usage.output_tokens)

            input_price = 0.75 / 1_000_000
            output_price = 4.50 / 1_000_000
            cost = (
                usage.input_tokens * input_price +
                usage.output_tokens * output_price
            )
            span.set_attribute("cost", cost)

            return response

    def rag(self, query):
        with tracer.start_as_current_span("rag") as span:
            span.set_attribute("query", query)

            search_results = self.search(query)
            prompt = self.build_prompt(query, search_results)
            response = self.llm(prompt)

            return response.output_text

In [3]:
from starter import index, client

rag_traced = RAGTraced(index=index, llm_client=client)

In [10]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_traced.rag(query)
print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0xe7fe3f153bd67d0cf4c573193a08061b",
        "span_id": "0x836bd9ab1ea2ba03",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xc21ead85e1fe8d57",
    "start_time": "2026-07-20T16:31:43.472388Z",
    "end_time": "2026-07-20T16:31:43.478297Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query": "How does the agentic loop keep calling the model until it stops?",
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "3dcf37d3-6934-478c-b8bc-83728b4c0246",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xe7fe3f153bd67d0cf

In [4]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult

class SQLiteSpanExporter(SpanExporter):
    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

**<h2> I had to restart the kernel here, as overriding TracerProvider is not allowed </h>**

In [5]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor

provider_sql = TracerProvider()
provider_sql.add_span_processor(SimpleSpanProcessor(SQLiteSpanExporter("traces.db")))
trace.set_tracer_provider(provider_sql)

tracer_sql = trace.get_tracer("llm-zoomcamp")

In [7]:
from rag_helper import RAGBase

class RAGTracedSQL(RAGBase):
    def search(self, query, num_results=5):
        with tracer_sql.start_as_current_span("search") as span:
            span.set_attribute("query", query)
            span.set_attribute("num_results", num_results)
            return super().search(query, num_results=num_results)

    def llm(self, prompt):
        with tracer_sql.start_as_current_span("llm") as span:
            span.set_attribute("prompt_length", len(prompt))
            response = super().llm(prompt)

            usage = response.usage
            span.set_attribute("input_tokens", usage.input_tokens)
            span.set_attribute("output_tokens", usage.output_tokens)

            input_price = 0.75 / 1_000_000
            output_price = 4.50 / 1_000_000
            cost = (
                usage.input_tokens * input_price +
                usage.output_tokens * output_price
            )
            span.set_attribute("cost", cost)

            return response

    def rag(self, query):
        with tracer_sql.start_as_current_span("rag") as span:
            span.set_attribute("query", query)

            search_results = self.search(query)
            prompt = self.build_prompt(query, search_results)
            response = self.llm(prompt)

            return response.output_text

In [8]:
rag_traced_SQL = RAGTracedSQL(index=index, llm_client=client)

In [15]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag_traced_SQL.rag(query)
print(answer)

It keeps calling the model inside a `while True` loop.

Each turn, it:
1. sends the full `messages` history to the model,
2. checks the response for any `function_call`,
3. runs the tool and appends the tool result to `messages`,
4. repeats.

It stops when a model response contains no function calls:

```python
if has_function_calls == False:
    break
```

So the loop ends when the model returns a final answer instead of asking for more tools.


In [10]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("traces.db")
df_spans = pd.read_sql_query("SELECT * FROM spans", conn)
df_spans

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search,1784566397444466000,1784566397449872000,NaN,NaN,NaN
1,llm,1784566397451533000,1784566399458583000,7111.0,98.0,0.005774
2,rag,1784566397444365000,1784566399459733000,NaN,NaN,NaN


In [11]:
df_spans["duration"] = df_spans["end_time"] - df_spans["start_time"]
df_spans

,name,start_time,end_time,input_tokens,output_tokens,cost,duration
0,search,1784566397444466000,1784566397449872000,NaN,NaN,NaN,5406000
1,llm,1784566397451533000,1784566399458583000,7111.0,98.0,0.005774,2007050000
2,rag,1784566397444365000,1784566399459733000,NaN,NaN,NaN,2015368000


In [16]:
df_spans2 = pd.read_sql_query("SELECT * FROM spans", conn)
df_spans2

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search,1784566397444466000,1784566397449872000,NaN,NaN,NaN
1,llm,1784566397451533000,1784566399458583000,7111.0,98.0,0.005774
2,rag,1784566397444365000,1784566399459733000,NaN,NaN,NaN
3,search,1784566576326065000,1784566576328213000,NaN,NaN,NaN
4,llm,1784566576329234000,1784566578151497000,7111.0,91.0,0.005743
5,rag,1784566576326010000,1784566578154515000,NaN,NaN,NaN
6,search,1784566581115586000,1784566581118464000,NaN,NaN,NaN
7,llm,1784566581119500000,1784566583782573000,7111.0,150.0,0.006008
8,rag,1784566581115542000,1784566583784637000,NaN,NaN,NaN
9,search,1784566584550712000,1784566584552868000,NaN,NaN,NaN
